# Responsible AI for 30-Day Hospital Readmission
## Patients with diabetes — discharge-time risk stratification

**Team:** _Add names_ · **Course:** AI for Healthcare · **Status:** Notebook skeleton, not a completed analysis

**Proposed question:** Among eligible live discharges involving patients with diabetes, can a calibrated model help prioritize limited post-discharge follow-up while making errors and subgroup disparities explicit?

This notebook follows the nine rubric steps in **2026_AI4HC_Session 5-6.pdf (pages 6–8)** and adapts the workflow—not the conclusions—from the [opioid-risk reference project](https://github.com/IE-ML-for-Healthcare/RAI_opioid_risk_prevention).

**How to use this skeleton:** Configuration runs as written. Analysis code is intentionally commented and labelled **STARTER TEMPLATE**; uncomment and complete it section by section. Running the skeleton does not download data, train a model, or produce results. Replace every *Complete* prompt with evidence from your own analysis. Do not present suggested hypotheses or illustrative capacity assumptions as findings.

**Scope:** Retrospective educational evaluation using historical US records. No clinical deployment, diagnosis, or demonstrated prevention of readmissions.


### Notebook map and rubric weights

| Step | Section | Weight |
|---|---|---:|
| 1 | Clinical problem framing and study objective | 10% |
| 2 | Data characterization and representativeness | 10% |
| 3 | Experimental design and splits | 5% |
| 4 | Pipelines, baselines and leakage controls | 10% |
| 5 | Metrics and discrimination analysis | 15% |
| 6 | Probability calibration and uncertainty | 8% |
| 7 | Threshold selection and decision analysis | 8% |
| 8 | Final test evaluation and generalization | 4% |
| 9 | Responsible AI analysis — RAI Toolbox | 30% |

Each section ends with a brief **So what?** for clinicians, patients, or decision-makers.


### Setup and reproducibility

The reference uses Python 3.10 and scikit-learn 1.5.1 with pandas, NumPy, plotting libraries, `responsibleai`, and `raiwidgets`; see its [environment.yml](https://github.com/IE-ML-for-Healthcare/RAI_opioid_risk_prevention/blob/main/environment.yml). Treat it as a starting point, not proof of compatibility on this machine. Confirm the RAI dashboard early, then pin the environment actually used.

**Complete:** Record interpreter/library versions and dataset checksum. The assignment also requires a README, environment.yml, and RAI presentation; those are separate deliverables, not supplied by this notebook skeleton. A utils.py is optional.


In [ ]:
from pathlib import Path

RANDOM_STATE = 42
DATA_DIR = Path("Data")  # Run Jupyter from the project directory.
DATA_PATH = DATA_DIR / "diabetic_data.csv"
MAPPING_PATH = DATA_DIR / "IDS_mapping.csv"
TARGET = "readmitted_30d"
PATIENT_ID = "patient_nbr"
DATA_URL = (
    "https://archive.ics.uci.edu/static/public/296/"
    "diabetes+130-us+hospitals+for+years+1999-2008.zip"
)

# Illustrative policy assumptions — justify before inspecting the final test.
ALERT_BUDGET_PER_1000 = 200
RECALL_FLOOR = 0.60


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import sklearn
#
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LogisticRegression
# from sklearn.dummy import DummyClassifier
# from sklearn.model_selection import GroupShuffleSplit
# from sklearn.metrics import (
#     average_precision_score, roc_auc_score, brier_score_loss,
#     confusion_matrix, precision_score, recall_score,
#     RocCurveDisplay, PrecisionRecallDisplay,
# )
# from sklearn.calibration import CalibrationDisplay
#
# print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)


## 1. Clinical problem framing and study objective — 10%

**Proposed intended use:** At discharge, rank eligible patients for additional nurse follow-up or medication-review outreach. The score supports review; it does not justify refusing care.

- **Population:** Inpatient encounters with a recorded diabetes diagnosis, a 1–14 day stay, laboratory tests, and administered medications, further restricted to the project's eligible discharge cohort.
- **Prediction time:** Discharge, after the current admission's permitted information is available.
- **Outcome:** Recorded readmission **<30 days**, versus `>30` or `NO`. Preserve the source definition; do not silently reinterpret this as ≤30 days or guaranteed capture of every readmission.
- **Decision:** Allocate a limited number of follow-up contacts.
- **False negative harm:** A patient who is subsequently readmitted is not prioritized.
- **False positive harm:** Unnecessary contact, patient burden, and staff time diverted from other patients.
- **Success criteria:** Compare to a prevalence baseline; assess PR performance, calibrated probabilities, workload, missed cases, and subgroup uncertainty. Do not promise an AUC or a reduction in readmissions.

**Complete:** Final inclusion/exclusion criteria, stakeholder, workflow, capacity rationale, and acceptable harms. Explain why a high-risk patient is not necessarily one who benefits most from the proposed intervention.

**So what?** _In two sentences: who would use the score, and what would change for a patient?_


## 2. Data characterization and representativeness — 10%

**Source:** [UCI Diabetes 130-US Hospitals for Years 1999–2008](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008). Download and extract `diabetic_data.csv` and `IDS_mapping.csv` into `Data/`. **License:** CC BY 4.0; cite Clore, Cios, DeShazo, and Strack (2014), DOI [10.24432/C5230J](https://doi.org/10.24432/C5230J).

The published dataset contains **101,766 encounters**, not 101,766 distinct patients, and 47 listed predictors. These are source characteristics, not your final analytic-cohort results.

**Complete these outputs:**
1. Data dictionary: type, clinical meaning, availability at discharge, missing-value encoding, intended role.
2. Cohort flow: raw encounters/patients → exclusions with counts → retained encounters/patients and positive cases.
3. Cohort table and plots: age bands, recorded race/gender, utilization, outcome prevalence, and missingness.
4. Label-quality and representativeness discussion: historical US inpatient sample, access/coding effects, incomplete observation of readmissions, and limits of transfer to present-day or Spanish practice.

**Cohort decisions:** Inspect discharge mappings for death, hospice, transfers, and other destinations. Decide which are eligible for the proposed follow-up service. Do not treat deaths as ordinary low-risk live discharges. Keep repeated eligible encounters if desired, but isolate patients across partitions.

**Missingness:** `?` is used for missing information. The literal category `None` in test-result fields can mean a test was not performed; do not silently convert every such value into generic missingness. Assess heavily missing `weight`, `payer_code`, and `medical_specialty` before selecting features.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# raw = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=["?"])
# required = {"encounter_id", PATIENT_ID, "readmitted", "discharge_disposition_id"}
# assert required.issubset(raw.columns)
# assert raw["readmitted"].isin(["<30", ">30", "NO"]).all()
# assert raw[PATIENT_ID].notna().all()
#
# raw[TARGET] = raw["readmitted"].eq("<30").astype("int8")
# print("Encounters:", len(raw))
# print("Patients:", raw[PATIENT_ID].nunique())
# print(raw["readmitted"].value_counts(dropna=False))
# print(raw.isna().mean().sort_values(ascending=False))
#
# # IDS_mapping.csv contains multiple mapping sections: inspect it before parsing.
# print(MAPPING_PATH.read_text())
# # Complete: construct `cohort` with documented eligibility rules and a flow table.
# # Freeze the cohort rules before evaluating candidate models.


### Data findings and clinical interpretation

**Complete:** Insert the cohort table, outcome/missingness plots, and inclusion flow. Report both encounter and unique-patient denominators. Initial source/cohort characterization is not permission to repeatedly inspect final-test patterns during model development; perform feature-selection EDA on training data after splitting.

**So what?** _Who is poorly represented, whose outcome might be missed, and where should this model not be used?_


## 3. Experimental design and splits — 5%

**Design:** Patient-disjoint training, validation, and final test partitions. Further split validation into **calibration** and **policy-selection** partitions, also by patient. This follows the rubric's validation-only calibration requirement while avoiding calibration and threshold selection on identical records.

**Proposed allocation by patients:** 70% training / 7.5% calibration / 7.5% policy validation / 15% test. Encounter proportions and prevalence will differ; this is group splitting, not stratification. Do not search random seeds for favorable metrics.

- Fit learned preprocessing and select/tune the model using training data only, with patient-group-aware CV if tuning.
- Fit probability calibration on calibration validation only.
- Select the operating threshold on policy validation only.
- Lock the full pipeline and threshold before final test evaluation.
- The public file does not provide suitable dates or hospital IDs for a true temporal/site split. Do not use encounter IDs as invented timestamps. This tests transfer to held-out patients within the historical source population, not future hospitals.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# def split_by_patient(frame, test_size):
#     splitter = GroupShuffleSplit(
#         n_splits=1, test_size=test_size, random_state=RANDOM_STATE
#     )
#     left, right = next(splitter.split(frame, groups=frame[PATIENT_ID]))
#     return frame.iloc[left].copy(), frame.iloc[right].copy()
#
# train_df, holdout_df = split_by_patient(cohort, test_size=0.30)
# validation_df, test_df = split_by_patient(holdout_df, test_size=0.50)
# calibration_df, policy_df = split_by_patient(validation_df, test_size=0.50)
#
# partitions = {
#     "train": train_df, "calibration": calibration_df,
#     "policy": policy_df, "test": test_df,
# }
# patient_sets = {name: set(frame[PATIENT_ID]) for name, frame in partitions.items()}
# for name, ids in patient_sets.items():
#     for other, other_ids in patient_sets.items():
#         if name != other:
#             assert ids.isdisjoint(other_ids)
#
# # Complete: save a reproducible split manifest keyed by encounter/patient ID.
# # Record split sizes; inspect development prevalence and event counts.
# # Reserve detailed test outcome analysis for section 8.


**So what?** _Explain how patient separation prevents an unrealistically optimistic evaluation. State what this split still cannot establish._


## 4. Pipelines, baselines and leakage controls — 10%

**Baseline:** Training-prevalence `DummyClassifier(strategy="prior")`. Compare a logistic-regression pipeline before considering extra complexity.

**Complete:** Define a deliberately small, clinically justified initial feature set. Keep audit attributes available even if excluded from model inputs. Treat integer-coded admission/discharge categories as categorical; preserve the ordered meaning of age bands if transforming them.

**Leakage checklist:**
- Exclude `readmitted`, the engineered target, `encounter_id`, and `patient_nbr` from predictors.
- Use only information available at the declared discharge decision time.
- Fit imputation, scaling, encoding, selection, and any resampling on training folds only.
- Group CV folds by patient. Do not randomly cross-validate encounters.
- Keep validation/test prevalence intact; do not balance evaluation data.
- Do not remove meaningful repeated patient encounters as if they were duplicate errors.
- If later moving prediction to admission, remove end-of-stay features and revisit the entire design.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# # Provisional small feature set; justify changes using training data only.
# numeric_features = [
#     "time_in_hospital", "num_lab_procedures", "num_procedures",
#     "num_medications", "number_outpatient", "number_emergency",
#     "number_inpatient", "number_diagnoses",
# ]
# categorical_features = [
#     "age", "race", "gender", "admission_type_id",
#     "admission_source_id", "A1Cresult", "insulin", "change", "diabetesMed",
# ]
# feature_columns = numeric_features + categorical_features
# assert not {TARGET, "readmitted", "encounter_id", PATIENT_ID} & set(feature_columns)
#
# numeric_pipeline = Pipeline([
#     ("impute", SimpleImputer(strategy="median")),
#     ("scale", StandardScaler()),
# ])
# categorical_pipeline = Pipeline([
#     ("impute", SimpleImputer(strategy="most_frequent")),
#     ("encode", OneHotEncoder(handle_unknown="ignore")),
# ])
# preprocessor = ColumnTransformer([
#     ("numeric", numeric_pipeline, numeric_features),
#     ("categorical", categorical_pipeline, categorical_features),
# ])
# model = Pipeline([
#     ("preprocess", preprocessor),
#     ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
# ])
#
# X_train, y_train = train_df[feature_columns], train_df[TARGET]
# X_cal, y_cal = calibration_df[feature_columns], calibration_df[TARGET]
# X_policy, y_policy = policy_df[feature_columns], policy_df[TARGET]
# baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
# model.fit(X_train, y_train)
# # Complete: document convergence, training-only model selection, and feature rationale.


**So what?** _Why is this baseline meaningful? Which predictors reflect illness versus access, documentation, or prior utilization? Coefficients are associations, not causal effects._


## 5. Metrics and discrimination analysis — 15%

**Primary discrimination metric:** Average precision (AP); report it explicitly rather than confusing it with trapezoidal PR AUC. Compare it with outcome prevalence. Also report ROC AUC and plot both curves.

**Before selecting a threshold:** Compare baseline and uncalibrated model on policy validation. Explore prespecified subgroups, reporting sample size and positive cases alongside performance. Do not choose the model from final-test results.

**Complete:** Development metric table, ROC/PR plots, subgroup discrimination with uncertainty, and discussion of why accuracy alone is insufficient. Threshold-specific patient-impact metrics belong in section 7.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# positive_index = list(model.classes_).index(1)
# p_policy_raw = model.predict_proba(X_policy)[:, positive_index]
# p_policy_baseline = baseline.predict_proba(X_policy)[:, list(baseline.classes_).index(1)]
#
# for name, scores in [("Prevalence baseline", p_policy_baseline), ("Logistic", p_policy_raw)]:
#     print(name, {
#         "prevalence": y_policy.mean(),
#         "average_precision": average_precision_score(y_policy, scores),
#         "roc_auc": roc_auc_score(y_policy, scores),
#     })
# RocCurveDisplay.from_predictions(y_policy, p_policy_raw)
# PrecisionRecallDisplay.from_predictions(y_policy, p_policy_raw)
# plt.show()
# # Complete: repeat metrics for prespecified cohorts; flag undefined metrics
# # for single-class groups rather than substituting misleading zeros.


**So what?** _Does the score improve ranking over the baseline? Which groups have too few positive cases to support confident comparisons?_


## 6. Probability calibration and uncertainty — 8%

**Goal:** A score of 0.20 should correspond approximately to a 20% recorded-readmission rate among comparable scored encounters—not a guarantee for an individual.

**Complete:** Freeze the fitted training model, fit a sigmoid calibrator on `X_cal, y_cal` only, then assess raw versus calibrated scores on `X_policy, y_policy`. Do not refit the base model on calibration data or calibrate on test.

**Version-sensitive implementation:** Use the supported prefit/frozen-estimator calibration API for the pinned scikit-learn version. In the reference's 1.5.1 environment, this is `CalibratedClassifierCV(model, method="sigmoid", cv="prefit")`; newer versions offer `FrozenEstimator`. Verify the actual environment before choosing the API.

**Outputs:** Reliability plots; Brier score; mean predicted probability versus observed prevalence; uncertainty intervals. Use patient-cluster bootstrap resampling when calculating intervals because encounters from one patient are correlated. Do not interpret Brier score alone as a pure measure of calibration.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# # Complete: create `calibrated_model` using the version-appropriate frozen API,
# # then fit ONLY on X_cal, y_cal.
#
# p_policy = calibrated_model.predict_proba(X_policy)[
#     :, list(calibrated_model.classes_).index(1)
# ]
# for name, scores in [("Raw", p_policy_raw), ("Calibrated", p_policy)]:
#     print(name, {
#         "brier": brier_score_loss(y_policy, scores),
#         "mean_probability": scores.mean(),
#         "observed_prevalence": y_policy.mean(),
#     })
#     CalibrationDisplay.from_predictions(y_policy, scores, n_bins=10, name=name)
# plt.show()
# # Complete: uncertainty method, patient-cluster resampling, and subgroup calibration.


**So what?** _Can the probabilities support workload planning? Does calibration improve reliability without implying that the intervention itself prevents readmission?_


## 7. Threshold selection and decision analysis — 8%

Compare at least **two methods on policy validation only**:

1. **Workload-constrained:** Capture the most positive cases while staying within the illustrative budget of 200 alerts per 1,000 eligible discharges.
2. **Recall-floor:** Achieve at least 60% recall while maximizing precision. These assumptions require justification; they are not clinical standards.
3. **Optional:** Sensitivity analysis using explicit false-positive/false-negative harm weights; label invented weights as illustrative rather than monetary estimates.

**Complete:** Enumerate thresholds over distinct scores plus boundary policies; define `score >= threshold`, tie handling, and deterministic tie-breaking. If no threshold meets joint constraints, report infeasibility rather than silently relaxing them. Select and justify one final operating policy before test access.

**Required table:** Threshold, TP/FP/FN/TN, precision, recall, specificity, alerts per 1,000, false alerts per 1,000, missed readmissions per 1,000, and estimated staff time under a stated minutes-per-contact assumption.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# threshold_rows = []
# # Infinity explicitly represents "alert nobody", including if a score equals 1.
# for threshold in np.r_[0.0, np.unique(p_policy), np.inf]:
#     predictions = (p_policy >= threshold).astype(int)
#     tn, fp, fn, tp = confusion_matrix(y_policy, predictions, labels=[0, 1]).ravel()
#     n = len(y_policy)
#     threshold_rows.append({
#         "threshold": threshold, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
#         "precision": tp / (tp + fp) if tp + fp else np.nan,
#         "recall": tp / (tp + fn) if tp + fn else np.nan,
#         "specificity": tn / (tn + fp) if tn + fp else np.nan,
#         "alerts_per_1000": 1000 * (tp + fp) / n,
#         "false_alerts_per_1000": 1000 * fp / n,
#         "missed_cases_per_1000": 1000 * fn / n,
#     })
# threshold_table = pd.DataFrame(threshold_rows)
# # Complete: select workload-constrained and recall-floor candidates separately.
# # Compare them, justify the chosen policy, and assign `locked_threshold`.
# # Save the model, feature schema, threshold, and decision rationale before test use.


**So what?** _For 1,000 discharges, how many calls would be made, how many would concern patients without recorded early readmission, and how many readmissions would be missed? Counts are NOT readmissions prevented._


## 8. Final test evaluation and generalization — 4%

**Gate:** Model, preprocessing, calibrator, feature set, threshold, and subgroup definitions are locked. This section is the first final-test performance evaluation.

**Complete:** One final report containing AP, ROC AUC, Brier score, reliability plot, confusion matrix, precision/recall/specificity, and per-1,000 patient-impact metrics at the locked threshold. Add patient-cluster uncertainty intervals and prespecified subgroup fairness checks.

Report whether the validation workload/recall assumptions hold on test. Do not alter the threshold to make them hold. Discuss age of the data, source-population limits, outcome capture, uncertainty, and absent external/temporal validation.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# X_test, y_test = test_df[feature_columns], test_df[TARGET]
# p_test = calibrated_model.predict_proba(X_test)[
#     :, list(calibrated_model.classes_).index(1)
# ]
# y_test_pred = (p_test >= locked_threshold).astype(int)
# print("Test AP:", average_precision_score(y_test, p_test))
# print("Test ROC AUC:", roc_auc_score(y_test, p_test))
# print("Test Brier score:", brier_score_loss(y_test, p_test))
# print("Locked-threshold confusion matrix:")
# print(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))
# # Complete: impact table, reliability plot, uncertainty, and subgroup report.
# # No fitting or threshold optimization on test.


**So what?** _Did the operating policy transfer to held-out patients? What evidence would still be needed before a prospective pilot?_


## 9. Responsible AI analysis — RAI Toolbox — 30%

Use **RAIInsights** and **ResponsibleAIDashboard** to examine the locked model. Verify dashboard compatibility early using development data. Use policy validation for iterative exploration; final-test findings are descriptive and must not feed back into this run's model or threshold.

**Input contract:** `rai_train` and `rai_eval` must contain model features plus `TARGET`, with compatible column names/types and no patient/encounter identifiers as model features. Ensure dashboard managers and the model see compatible representations. If imputation is needed for dashboard components, reuse training-fitted transformations; do not refit preprocessing on evaluation data. Keep clinically readable categories rather than displaying unexplained one-hot columns.

**Threshold warning:** A calibrated classifier's `.predict()` generally uses a default cutoff, not `locked_threshold`. Confirm which threshold the dashboard uses. Report policy fairness/error metrics at the locked threshold separately; do not label default-cutoff dashboard findings as your selected policy's results.


In [ ]:
# STARTER TEMPLATE — uncomment when completing this section.
# from responsibleai import RAIInsights
# from raiwidgets import ResponsibleAIDashboard
#
# # Complete: prepare compatible `rai_train` and `rai_eval` dataframes,
# # using training and policy-validation data respectively, and verify the
# # RAI-supported model interface for the fitted calibrated pipeline.
# rai = RAIInsights(
#     model=calibrated_model,
#     train=rai_train,
#     test=rai_eval,
#     target_column=TARGET,
#     task_type="classification",
#     categorical_features=categorical_features,
# )
# rai.explainer.add()
# rai.error_analysis.add()
# # Complete: configure counterfactuals and causal analysis as discussed below.
# rai.compute()
# ResponsibleAIDashboard(rai)


### 9.1 Data analysis and cohort exploration

**Complete:** Create at least one clinically meaningful cohort, record its exact rule and size, and compare feature distributions, missingness, and outcome prevalence. Candidate hypotheses: older patients, prior inpatient use, and patients with no HbA1c test recorded. These are questions to investigate, not findings.

**Evidence:** _Dashboard screenshot + cohort definition + non-trivial observation._

**So what?** _Which clinical or data-collection difference might explain what you observed?_


### 9.2 Model overview and fairness

**Complete:** Compare recall/FNR, FPR, precision, selection rate, and calibration across recorded race, gender, and age cohorts. Include denominators and positive counts; retain unknown categories transparently. Discuss uncertainty and competing fairness objectives rather than declaring fairness from one metric.

Investigate whether disparities reflect access or label capture as well as prediction errors. Subgroup thresholds are an optional mitigation requiring ethical/legal review, not an automatic fix.

**Evidence:** _Dashboard screenshot + locked-policy subgroup table + limitation._

**So what?** _Who bears more missed-case or unnecessary-contact burden?_


### 9.3 Error analysis

**Complete:** Use the error tree/heatmap to identify a concentrated failure cohort, inspect false negatives and false positives, and compare it with the overall population. Treat newly discovered cohorts as exploratory; quantify their size and avoid overclaiming from small groups.

**Evidence:** _Screenshot + exact cohort rule + error rates/counts + plausible explanation._

**So what?** _What targeted review, data improvement, or monitoring would address this failure?_


### 9.4 Feature importance

**Complete:** Show global importance and local explanations for representative correct and incorrect predictions. Consider correlated features and proxies: prior utilization can reflect illness, access, and local practice. Explain the prediction, not a causal mechanism.

**Evidence:** _Global plot + local examples + clinically readable interpretation._

**So what?** _Does the model rely on information clinicians could reasonably use at discharge?_


### 9.5 Counterfactual analysis

**Complete:** Configure the counterfactual manager with clinically defensible permitted ranges and features-to-vary. Keep race, gender, age, diagnoses, and historical visit counts fixed for actionable discharge-time scenarios. Do not suggest reducing recorded prior admissions or changing identity to lower risk.

Medication or testing changes may be model scenarios, but they are not safe treatment recommendations without clinical review and causal evidence. If no defensible actionable counterfactual exists with the chosen features, demonstrate the limitation rather than manufacturing recourse. Check whether the counterfactual class boundary matches the selected policy threshold.

**Evidence:** _Screenshot + permitted changes + plausibility/immutability assessment._

**So what?** _What does the scenario reveal about the model, and what does it NOT tell a clinician to do?_


### 9.6 Causal analysis

**Complete:** Specify a treatment/exposure, outcome, time ordering, estimand, and a causal diagram before configuring `rai.causal`. One question to assess is whether HbA1c testing during admission is associated with later readmission under an explicit adjustment strategy; testing is not randomized and illness severity may affect both testing and readmission.

List plausible confounders, treatment overlap/positivity, selection effects, and unavailable information. Do not use downstream variables indiscriminately as adjustment covariates. If proceeding with the dashboard demonstration, label estimates as exploratory and assumption-dependent. Explain why this dataset cannot establish that follow-up calls prevent readmission: it does not supply the required intervention comparison.

**Evidence:** _Configured analysis and screenshot where estimable + assumptions + identification limitations. If estimation is unsupported, document the specific reason and discuss how to meet the rubric with the instructor rather than inventing an effect._

**So what?** _Which causal claim is unsupported, and what prospective data or study would be needed?_


### 9.7 Deployment risks, mitigations, and monitoring

**Complete after analysis:** Identify the top three risks supported by your results. Do not pre-fill these as observed findings.

| Observed risk and evidence | Affected patients | Proposed mitigation | Expected effect and basis | Monitoring measure / alert trigger | Owner and response |
|---|---|---|---|---|---|
| _Complete risk 1_ | | | | | |
| _Complete risk 2_ | | | | | |
| _Complete risk 3_ | | | | | |

Quantify expected mitigation effects only when measured, or label them as assumptions with a range. Include policy pros/cons, patient consequences, and limits of automated decision-making.

**Clinician-ready conclusion:** _Three brief recommendations supported by evidence; one explicit non-recommendation; next validation step._


## Submission checklist and references

- All nine sections contain outputs, learnings, and brief clinical implications.
- Split manifest, seed, environment versions, data provenance/license, and cohort rules are recorded.
- Calibration uses validation only; threshold selection uses separate policy validation; final test remains locked until section 8.
- At least two threshold methods are compared with workload and patient-impact counts.
- RAI evidence covers data analysis, model overview/fairness, errors, importance, counterfactuals, and causal analysis with appropriate limitations.
- Export selected dashboard screenshots for the required presentation.
- Write the separate README (purpose, structure, install, quickstart, data, evaluation/thresholds, RAI/reproducibility/license) and environment.yml. Extract repeated helpers into utils.py if useful.
- Replace all *Complete* prompts and unfinished starter templates before submission; this skeleton itself is not the finished assignment.

### References
1. Course rubric: *2026_AI4HC_Session 5-6.pdf*, pages 6–8; additional guidance on pages 9–11.
2. [Reference notebook repository: RAI opioid risk prevention](https://github.com/IE-ML-for-Healthcare/RAI_opioid_risk_prevention). Its synthetic OUD findings do not transfer to this dataset.
3. [Diabetes 130-US Hospitals dataset and variable documentation](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008), Clore et al. (2014), CC BY 4.0.
4. Strack et al. (2014), [Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records](https://doi.org/10.1155/2014/781670).
